In [1]:
# TASK 4: WorkingMemLoad — Working Memory Under Interference
# CogExec-EF Executive Functions Benchmark
#
# Cognitive faculty : Executive Functions → Working Memory
# Research basis    : Baddeley & Hitch (1974) working memory model —
#                     specifically proactive interference from distractor content
# What it isolates  : Can the model retain target facts in working memory
#                     while processing a long thematically adjacent distractor
#                     passage — then recall the ORIGINAL facts?
# Design            : Three-turn stateful conversation (n_jobs=1 required)
#                     Turn 1 → memorise target facts
#                     Turn 2 → read distractor passage (interference)
#                     Turn 3 → recall question about original facts
# Key features      : interference_level (low/medium/high)
#                     interference_mechanism (semantic_priming, number_substitution etc)
#                     foil_2 — second engineered wrong answer
# Scoring           : tuple[int, int] → (passes, 2)

import kaggle_benchmarks as kbench
import pandas as pd
import re

df = pd.read_csv("/kaggle/input/datasets/jaytalwar2005/cogexec-ef-benchmark-data/workingmemload_150_FINAL_v3.csv")
print(f"Loaded {len(df)} rows")
print(f"Difficulty:            {df['difficulty'].value_counts().to_dict()}")
print(f"Category:              {df['category'].value_counts().to_dict()}")
print(f"Interference level:    {df['interference_level'].value_counts().to_dict()}")
print(f"Interference mechanism:{df['interference_mechanism'].value_counts().to_dict()}")

def safe_pattern(text: str) -> str:
    return re.escape(str(text).strip())

Loaded 125 rows
Difficulty:            {'medium': 54, 'hard': 38, 'easy': 33}
Category:              {'simple_recall': 20, 'association_recall': 19, 'arithmetic_on_facts': 16, 'multi_fact_recall': 16, 'ordered_sequence': 15, 'conditional_recall': 15, 'interference_heavy': 14, 'source_monitoring': 10}
Interference level:    {'medium': 49, 'high': 43, 'low': 33}
Interference mechanism:{'number_substitution': 20, 'semantic_priming': 19, 'computation_plus_retention': 16, 'multi_element_binding': 16, 'positional_confusion': 15, 'filter_application': 15, 'direct_value_match': 14, 'source_attribution': 10}


In [2]:
from kaggle_benchmarks import llms
import kaggle_benchmarks as kbench

llm1 = kbench.llm   # Gemini Flash (baseline)

llm2 = llms.get("google/gemma-4-26b-a4b")   # weak

llm3 = llms.get("openai/gpt-5.4-mini-2026-03-17")   # mid

llm4 = llms.get("google/gemini-3.1-pro-preview")    # strong

llm5 = llms.get("anthropic/claude-sonnet-4-6@default")   # very strong

llm6 = llms.get("deepseek-ai/deepseek-r1-0528")    # reasoning-heavy

all_models = [llm1, llm2, llm3, llm4, llm5, llm6]

for i, m in enumerate(all_models, 1):
    status = "Loaded" if m else "Failed"
    print(f"llm{i}: {m} --> {status}")

llm1: 🤖 deepseek-ai/deepseek-r1-0528 --> Loaded
llm2: 🤖 google/gemma-4-26b-a4b --> Loaded
llm3: 🤖 openai/gpt-5.4-mini-2026-03-17 --> Loaded
llm4: 🤖 google/gemini-3.1-pro-preview --> Loaded
llm5: 🤖 anthropic/claude-sonnet-4-6@default --> Loaded
llm6: 🤖 deepseek-ai/deepseek-r1-0528 --> Loaded


In [3]:
# ── Task definition ───────────────────────────────────────────────────────────
@kbench.task(name="working_mem_load", version=1)
def working_mem_load(
    llm,
    id: int,
    facts_to_remember: str,
    distractor_topic: str,
    distractor_text: str,
    recall_question: str,
    correct_answer: str,
    wrong_answer: str,
    foil_2: str,
    answer_choices: str,
    category: str,
    difficulty: str,
    interference_level: str,
    interference_mechanism: str,
    explanation: str,
    answer_format: str,
    scoring_method: str,
    ef_component: str,
) -> tuple[int, int]:
    """Working memory under interference: memorise facts, survive a distractor passage, recall correctly. Three-turn stateful conversation. Difficulty: easy/medium/hard."""

    # Turn 1: load target facts into working memory
    llm.prompt(
        "Memorise the following facts carefully. "
        "You will be tested on them later — after reading something else.\n\n"
        f"FACTS TO REMEMBER:\n{facts_to_remember}\n\n"
        "Confirm by saying: 'Facts noted.'"
    )

    # Turn 2: distractor injection — proactive interference
    # interference_mechanism documents exactly HOW the distractor primes wrong_answer
    llm.prompt(
        f"Now read the following passage about {distractor_topic} "
        "and summarise it in one sentence.\n\n"
        f"{distractor_text}"
    )

    # Turn 3: recall test — must retrieve from working memory, not distractor
    response = llm.prompt(
        "Now answer this question about the FACTS YOU MEMORISED at the start — "
        "NOT about the passage you just read.\n\n"
        f"Question: {recall_question}\n\n"
        "Reply with ONLY the answer, nothing else."
    )

    passes = 0
    total = 2

    # Assertion 1: correct answer must be recalled
    r1 = kbench.assertions.assert_contains_regex(
        rf"(?i){safe_pattern(correct_answer)}",
        response,
        expectation=f"Must recall the correct fact: '{correct_answer}' (interference_level: {interference_level}, mechanism: {interference_mechanism})",
    )
    if r1.passed:
        passes += 1

    # Assertion 2: distractor-primed wrong answer must NOT appear
    if safe_pattern(wrong_answer).lower() != safe_pattern(correct_answer).lower():
        r2 = kbench.assertions.assert_not_contains_regex(
            rf"(?i)\b{safe_pattern(wrong_answer)}\b",
            response,
            expectation=(
                f"Must NOT be misled by distractor into answering '{wrong_answer}'. "
                f"Interference mechanism: {interference_mechanism}. "
                f"Explanation: {explanation}"
            ),
        )
        if r2.passed:
            passes += 1

    return passes, total

In [4]:
# ── Smoke test ────────────────────────────────────────────────────────────────
print("\n── Smoke test (row 0) ──")
smoke = df.iloc[0]
run = working_mem_load.run(
    llm=kbench.llm,
    id=int(smoke["id"]),
    facts_to_remember=smoke["facts_to_remember"],
    distractor_topic=smoke["distractor_topic"],
    distractor_text=smoke["distractor_text"],
    recall_question=smoke["recall_question"],
    correct_answer=smoke["correct_answer"],
    wrong_answer=smoke["wrong_answer"],
    foil_2=smoke["foil_2"],
    answer_choices=smoke["answer_choices"],
    category=smoke["category"],
    difficulty=smoke["difficulty"],
    interference_level=smoke["interference_level"],
    interference_mechanism=smoke["interference_mechanism"],
    explanation=smoke["explanation"],
    answer_format=smoke["answer_format"],
    scoring_method=smoke["scoring_method"],
    ef_component=smoke["ef_component"],
)
print(f"Result: {run.result}  |  Passed: {run.passed}")
print("Smoke test complete")


── Smoke test (row 0) ──


Result: (1, 2)  |  Passed: False
Smoke test complete


In [5]:
# ── Multi-model evaluation ────────────────────────────────────────────────────
# IMPORTANT: n_jobs=1 — this is a 3-turn STATEFUL conversation
print("\n── Multi-model evaluation ──")
runs = working_mem_load.evaluate(
    llm=all_models,
    evaluation_data=df,
    n_jobs=1,
    max_attempts=3,
    retry_delay=5,
)


── Multi-model evaluation ──


In [6]:
# ── Results ───────────────────────────────────────────────────────────────

results_df = runs.as_dataframe()

# Convert (passes, total) → score
results_df["score"] = results_df["result"].apply(
    lambda x: x[0] / x[1] if isinstance(x, tuple) and x[1] > 0 else float(x)
)

# Clean model names
results_df["model_name"] = results_df["llm"].apply(lambda x: str(x))

# ── BASIC STATS ───────────────────────────────────────────────────────────

print(f"\nTotal runs    : {len(results_df)}")
print(f"Overall score : {results_df['score'].mean():.3f}")

# ── BREAKDOWN ─────────────────────────────────────────────────────────────

print("\nScore by difficulty:")
print(results_df.groupby("difficulty")["score"].mean().round(3))

print("\nScore by category:")
print(results_df.groupby("category")["score"].mean().round(3))

# Only if column exists
if "ef_component" in results_df.columns:
    print("\nScore by ef_component:")
    print(results_df.groupby("ef_component")["score"].mean().round(3))

print("\nScore by model:")
print(results_df.groupby("model_name")["score"].mean().round(3))

# ── MODEL COMPARISON TABLE───────────────────

print("\n── Model comparison (table) ──")

pivot_df = results_df.pivot_table(
    index="id",
    columns="model_name",
    values="score"
)

print(pivot_df.round(3))


results_df.to_csv("final_results.csv", index=False)
pivot_df.to_csv("model_comparison.csv")

print("\nResults saved as CSV files")


Total runs    : 749
Overall score : 0.721

Score by difficulty:
difficulty
easy      0.735
hard      0.704
medium    0.724
Name: score, dtype: float64

Score by category:
category
arithmetic_on_facts    0.729
association_recall     0.746
conditional_recall     0.561
interference_heavy     0.827
multi_fact_recall      0.663
ordered_sequence       0.733
simple_recall          0.800
source_monitoring      0.667
Name: score, dtype: float64

Score by ef_component:
ef_component
working_memory_binding                    0.746
working_memory_conditional_filter         0.561
working_memory_interference_resistance    0.827
working_memory_maintenance                0.800
working_memory_manipulation               0.729
working_memory_multiple_bindings          0.663
working_memory_sequential                 0.733
working_memory_source_attribution         0.667
Name: score, dtype: float64

Score by model:
model_name
🤖 anthropic/claude-sonnet-4-6@default    0.740
🤖 deepseek-ai/deepseek-r1-0528     